# Legal PDF Preprocessor — 26,703 PDFs
**Run every cell top to bottom. Do NOT skip any cell.**

- Input : `MyDrive/supreme_court_judgments/` (subfolders by year, .PDF extension)
- Output: `MyDrive/L_P/docs/doc_XXXXX.json` — one JSON per PDF

In [2]:
# ── CELL 1 ── Install dependencies
!pip install pdfplumber tqdm --quiet
print('✓ Done')

✓ Done


In [3]:
# ── CELL 2 ── Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
print('\n✓ Drive mounted')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✓ Drive mounted


In [7]:
# ── CELL 3 ── Set paths and verify PDF count
import os
from pathlib import Path

INPUT_DIR   = '/content/drive/MyDrive/supreme_court_judgments'
OUTPUT_DIR  = '/content/drive/MyDrive/L_P'
NUM_WORKERS = 2   # Free Colab=2, Colab Pro=4

os.makedirs(f'{OUTPUT_DIR}/docs', exist_ok=True)

# Case-insensitive count — matches both .pdf and .PDF
all_pdfs = [p for p in Path(INPUT_DIR).rglob('*') if p.suffix.lower() == '.pdf']

print(f'INPUT_DIR   : {INPUT_DIR}')
print(f'OUTPUT_DIR  : {OUTPUT_DIR}')
print(f'PDFs found  : {len(all_pdfs):,}')

if len(all_pdfs) == 0:
    print('\n⚠ No PDFs found — check folder name')
else:
    print(f'Sample      : {[p.name for p in all_pdfs[:3]]}')
    print('\n✓ Ready to run!')

INPUT_DIR   : /content/drive/MyDrive/supreme_court_judgments
OUTPUT_DIR  : /content/drive/MyDrive/L_P
PDFs found  : 26,703
Sample      : ['J_A_Naiksatam_vs_Prothonotary_Senior_Master_High_on_7_October_2004_1.PDF', 'Distt_Registrar_Collector_vs_Canara_Bank_Etc_on_1_November_2004_1.PDF', 'Green_View_Tea_And_Industries_vs_Collector_Golaghat_Assam_And_Anr_on_17_February_2004_1.PDF']

✓ Ready to run!


In [8]:
%%writefile /content/preprocess_dataset.py
import os, re, sys, json, time, logging, argparse, traceback, unicodedata
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from collections import defaultdict

try:
    from tqdm import tqdm
except ImportError:
    class tqdm:
        def __init__(self, it=None, total=None, **kw):
            self._it = it or []; self._n = 0; self._total = total or 0
        def __iter__(self):
            for x in self._it:
                self._n += 1
                if self._n % 500 == 0:
                    print(f'  ...{self._n}/{self._total}')
                yield x
        def update(self, n=1): pass
        def set_postfix(self, **kw): pass
        def close(self): pass

try:
    import pdfplumber
except ImportError:
    print('ERROR: pip install pdfplumber'); sys.exit(1)

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger(__name__)

MIN_TEXT_CHARS = 200
MIN_WORDS_DOC  = 50
MIN_WORDS_SENT = 4

SECTION_PATTERNS = {
    'FACTS': re.compile(
        r'^\s*(facts?|background|factual\s+background|brief\s+facts?|'
        r'statement\s+of\s+facts?|facts\s+of\s+the\s+case)\s*[:\.\-]?\s*$', re.I|re.M),
    'ISSUES': re.compile(
        r'^\s*(issues?\s+(?:raised|framed|for\s+(?:determination|consideration))|'
        r'questions?\s+of\s+law|points?\s+(?:of|for)\s+(?:determination|consideration)|'
        r'issues?)\s*[:\.\-]?\s*$', re.I|re.M),
    'ARGUMENTS': re.compile(
        r'^\s*(arguments?|submissions?|contentions?|'
        r'submissions?\s+(?:of\s+)?(?:counsel|parties|appellant|respondent|petitioner)|'
        r'counsel\s+(?:for|of))\s*[:\.\-]?\s*$', re.I|re.M),
    'REASONING': re.compile(
        r'^\s*(reasoning|analysis|discussion|findings?|ratio|ratio\s+decidendi|'
        r'observations?|held|courts?\s+(?:view|opinion|reasoning)|'
        r'our\s+(?:view|analysis|opinion))\s*[:\.\-]?\s*$', re.I|re.M),
    'JUDGMENT': re.compile(
        r'^\s*(judgment|judgement|order|decree|decision|conclusion|result|disposal|'
        r'operative\s+(?:part|portion)|for\s+(?:the\s+)?(?:above\s+)?reasons?|'
        r'in\s+(?:the\s+)?result|accordingly)\s*[:\.\-]?\s*$', re.I|re.M),
    'STATUTES': re.compile(
        r'^\s*(statutes?|(?:relevant\s+)?(?:statutory\s+)?provisions?|'
        r'(?:relevant\s+)?(?:applicable\s+)?law)\s*[:\.\-]?\s*$', re.I|re.M),
}

BOILERPLATE = [
    re.compile(r'indian\s+kanoon\s*[-]?\s*\S*', re.I),
    re.compile(r'take\s+notes\s*\|?\s*print',    re.I),
    re.compile(r'try\s+out\s+our\s+premium',     re.I),
    re.compile(r'equivalent\s+citations?\s*:',    re.I),
    re.compile(r'^\s*page\s+\d+\s+of\s+\d+\s*$', re.I|re.M),
    re.compile(r'^\s*\d+\s*$',                   re.M),
    re.compile(r'^\s*[-]{3,}\s*$',               re.M),
]

HYPHEN_RE = re.compile(r'(\w)-\n(\w)')
SOFTBR_RE = re.compile(r'(?<![.?!\n])\n(?=[a-z])')

NER_PATTERNS = {
    'SECTION': re.compile(
        r'\b(?:[Ss]ection[s]?\s+\d+[\w()]*|[Aa]rticle[s]?\s+\d+[\w()]*|'
        r'[Ss]ec\.?\s*\d+[\w()]*|[Aa]rt\.?\s*\d+[\w()]*|[Rr]ule[s]?\s+\d+[\w()]*)\b'),
    'ACT': re.compile(
        r'\b(?:Indian\s+Penal\s+Code|Indian\s+Contract\s+Act|Evidence\s+Act|'
        r'Transfer\s+of\s+Property\s+Act|Specific\s+Relief\s+Act|'
        r'Civil\s+Procedure\s+Code|Criminal\s+Procedure\s+Code|'
        r'Constitution\s+of\s+India|Limitation\s+Act|Information\s+Technology\s+Act|'
        r'Consumer\s+Protection\s+Act|Motor\s+Vehicles\s+Act|'
        r'Negotiable\s+Instruments\s+Act|Companies\s+Act|Income\s+Tax\s+Act|'
        r'Arbitration\s+Act|Prevention\s+of\s+Corruption\s+Act|NDPS\s+Act|'
        r'Right\s+to\s+Information\s+Act|RTI\s+Act|'
        r'[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s+Act(?:,?\s+\d{4})?|'
        r'I\.?P\.?C\.?|C\.?r\.?P\.?C\.?|C\.?P\.?C\.?)\b'),
    'CITATION': re.compile(
        r'\(?\d{4}\)?\s+\d+\s+(?:SCC|AIR|SCR|HC|BLR|MLJ|CLJ)\s+\d+'),
    'PARTY': re.compile(
        r'\b(?:appellant|respondent|petitioner|plaintiff|defendant|'
        r'accused|complainant|claimant)(?:\s+No\.?\s*\d+)?', re.I),
    'JUDGE': re.compile(
        r'\b(?:Justice\s+[A-Z][a-zA-Z\s.]+|[A-Z][a-z]+\s+(?:CJ|J\.?))\b'),
}

_LEGAL_TERMS = {
    'mens rea','actus reus','ratio decidendi','obiter dicta','locus standi',
    'res judicata','ab initio','ex parte','inter alia','prima facie',
    'suo motu','sub judice','ultra vires','habeas corpus','mandamus',
    'certiorari','injunction','specific performance','force majeure',
    'breach of contract','tortious liability','vicarious liability',
    'res ipsa loquitur','frustration of contract','burden of proof',
    'beyond reasonable doubt','natural justice','audi alteram partem',
    'legitimate expectation','promissory estoppel',
}
LEGAL_TERM_RE = re.compile(
    r'\b(' + '|'.join(re.escape(t) for t in sorted(_LEGAL_TERMS, key=len, reverse=True)) + r')\b',
    re.I)

_ABBREV_RE = re.compile(
    r'\b(Mr|Mrs|Ms|Dr|Prof|Vs|vs|No|Art|Sec|Cl|Sch|J|CJ|JJ|HC|SC|'
    r'i\.e|e\.g|etc|viz|ibid|cf|p|pp|ed|vol|Vol)\.', re.I)


def extract_pdf(path):
    pages = []
    try:
        with pdfplumber.open(path) as pdf:
            for pg in pdf.pages:
                try:
                    t = pg.extract_text()
                    if t and t.strip():
                        pages.append(t)
                except Exception:
                    pass
    except Exception:
        return ''
    return '\n'.join(pages)


def clean_text(raw):
    t = unicodedata.normalize('NFKC', raw)
    t = HYPHEN_RE.sub(r'\1\2', t)
    t = SOFTBR_RE.sub(' ', t)
    lines = [l for l in t.split('\n') if not any(p.search(l) for p in BOILERPLATE)]
    t = '\n'.join(lines)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]{2,}', ' ', t)
    return '\n'.join(l.rstrip() for l in t.split('\n')).strip()


def detect_sections(text):
    cur = 'PREAMBLE'
    buf = []
    bkts = defaultdict(list)
    for line in text.split('\n'):
        matched = None
        for lbl, pat in SECTION_PATTERNS.items():
            if pat.match(line):
                matched = lbl
                break
        if matched:
            bkts[cur].extend(buf); buf = []; cur = matched
        else:
            buf.append(line)
    bkts[cur].extend(buf)
    return {k: ' '.join(v).strip() for k, v in bkts.items() if ' '.join(v).strip()}


def segment_sentences(text):
    protected = _ABBREV_RE.sub(lambda m: m.group(0).replace('.', '##DOT##'), text)
    raw = re.split(r'(?<=[.!?])\s+(?=[A-Z\"\(])', protected)
    return [s.replace('##DOT##', '.').strip() for s in raw if len(s.split()) >= MIN_WORDS_SENT]


def extract_entities(text):
    found = defaultdict(set)
    for lbl, pat in NER_PATTERNS.items():
        for m in pat.finditer(text):
            found[lbl].add(m.group(0).strip().lower())
    for m in LEGAL_TERM_RE.finditer(text):
        found['LEGAL_TERM'].add(m.group(0).strip().lower())
    return {k: sorted(v) for k, v in found.items()}


def process_one(args):
    pdf_path, doc_id = args
    res = {'doc_id': doc_id, 'filename': os.path.basename(pdf_path),
           'filepath': str(pdf_path), 'status': 'error'}
    try:
        raw = extract_pdf(pdf_path)
        if len(raw.strip()) < MIN_TEXT_CHARS:
            res.update({'status': 'empty', 'error': f'Only {len(raw.strip())} chars'})
            return res
        cl = clean_text(raw)
        if len(cl.split()) < MIN_WORDS_DOC:
            res.update({'status': 'empty', 'error': f'Only {len(cl.split())} words'})
            return res
        sec_blobs = detect_sections(cl)
        sec_sents = {k: segment_sentences(v) for k, v in sec_blobs.items() if segment_sentences(v)}
        sentences = []
        idx = 0
        for lbl, sents in sec_sents.items():
            for s in sents:
                sentences.append({'id': idx, 'text': s, 'section': lbl, 'tokens': len(s.split())})
                idx += 1
        entities     = extract_entities(cl)
        total_tokens = sum(s['tokens'] for s in sentences)
        avg_tokens   = round(total_tokens / len(sentences), 2) if sentences else 0
        tok_per_sec  = {k: sum(s['tokens'] for s in sentences if s['section'] == k) for k in sec_sents}
        stats = {
            'word_count':              len(cl.split()),
            'char_count':              len(cl),
            'total_tokens':            total_tokens,
            'avg_tokens_per_sentence': avg_tokens,
            'tokens_per_section':      tok_per_sec,
            'num_sentences':           len(sentences),
            'num_sections':            len(sec_sents),
            'sections_found':          list(sec_sents.keys()),
            'section_counts':          {k: len(v) for k, v in sec_sents.items()},
            'has_facts':               'FACTS'     in sec_sents,
            'has_reasoning':           'REASONING' in sec_sents,
            'has_judgment':            'JUDGMENT'  in sec_sents,
            'entity_counts':           {k: len(v) for k, v in entities.items()},
        }
        res.update({'status': 'ok', 'stats': stats, 'sections': sec_sents,
                    'sentences': sentences, 'entities': entities})
    except Exception as e:
        res.update({'status': 'error', 'error': str(e), 'trace': traceback.format_exc()})
    return res


def load_checkpoint(path):
    if not os.path.exists(path):
        return set()
    with open(path) as f:
        done = {l.strip() for l in f if l.strip()}
    log.info(f'  Checkpoint: {len(done):,} docs already done — skipping')
    return done


def save_checkpoint(path, doc_id):
    with open(path, 'a') as f:
        f.write(doc_id + '\n')


def save_json(rec, docs_dir):
    out = os.path.join(docs_dir, f"{rec['doc_id']}.json")
    with open(out, 'w', encoding='utf-8') as f:
        json.dump(rec, f, ensure_ascii=False, indent=2)


def find_pdfs(input_dir, max_docs):
    # ✅ KEY FIX: case-insensitive match — finds both .pdf AND .PDF
    pdfs = sorted([p for p in Path(input_dir).rglob('*') if p.suffix.lower() == '.pdf'])
    log.info(f'  PDFs found : {len(pdfs):,}')
    if max_docs:
        pdfs = pdfs[:max_docs]
        log.info(f'  Limited to : {max_docs}')
    return pdfs


def compute_aggregate(ok_stats, n_ok, n_em, n_er, elapsed):
    if not ok_stats:
        return {'error': 'no successful docs', 'n_ok': n_ok, 'n_em': n_em, 'n_er': n_er}
    wc  = [s['word_count']    for s in ok_stats]
    sc  = [s['num_sentences'] for s in ok_stats]
    tok = [s['total_tokens']  for s in ok_stats]
    cov = {}
    for lbl in ['FACTS','ISSUES','ARGUMENTS','REASONING','JUDGMENT','STATUTES']:
        c = sum(1 for s in ok_stats if lbl in s.get('sections_found', []))
        cov[lbl] = {'docs': c, 'percent': round(100 * c / n_ok, 1) if n_ok else 0}
    return {
        'total_pdfs':              n_ok + n_em + n_er,
        'successful':              n_ok,
        'empty_or_short':          n_em,
        'errors':                  n_er,
        'processing_time_seconds': round(elapsed, 1),
        'docs_per_second':         round((n_ok + n_em + n_er) / elapsed, 2) if elapsed else 0,
        'word_count':  {'mean': round(sum(wc)/len(wc)), 'median': sorted(wc)[len(wc)//2],
                        'min': min(wc), 'max': max(wc)},
        'tokens':      {'total_corpus': sum(tok), 'mean_per_doc': round(sum(tok)/len(tok)),
                        'median_per_doc': sorted(tok)[len(tok)//2],
                        'min_per_doc': min(tok), 'max_per_doc': max(tok)},
        'sentences_per_doc': {'mean': round(sum(sc)/len(sc)), 'min': min(sc),
                              'max': max(sc), 'total': sum(sc)},
        'section_coverage': cov,
    }


def run_preprocessing(input_dir, output_dir, workers, max_docs):
    docs_dir = os.path.join(output_dir, 'docs')
    ckpt     = os.path.join(output_dir, 'checkpoint.txt')
    errs     = os.path.join(output_dir, 'errors.jsonl')
    stats_f  = os.path.join(output_dir, 'stats.json')
    os.makedirs(docs_dir, exist_ok=True)

    log.info('=' * 60)
    log.info('  LEGAL DATASET PREPROCESSOR')
    log.info('=' * 60)
    log.info(f'  Input   : {input_dir}')
    log.info(f'  Output  : {output_dir}')
    log.info(f'  Workers : {workers}')

    pdfs = find_pdfs(input_dir, max_docs)
    if not pdfs:
        log.error('No PDFs found. Check input_dir.')
        return

    done = load_checkpoint(ckpt)
    jobs = [(str(p), f'doc_{i:05d}') for i, p in enumerate(pdfs) if f'doc_{i:05d}' not in done]
    log.info(f'  To process : {len(jobs):,}')
    log.info('=' * 60)

    if not jobs:
        log.info('All documents already processed.')
        return

    t0 = time.time()
    n_ok = n_em = n_er = 0
    ok_stats = []

    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = {pool.submit(process_one, j): j for j in jobs}
        pbar = tqdm(as_completed(futures), total=len(jobs), desc='Preprocessing', unit='doc')
        for future in pbar:
            job = futures[future]
            try:
                rec = future.result(timeout=120)
            except Exception as e:
                rec = {'doc_id': job[1], 'filename': os.path.basename(job[0]),
                       'status': 'error', 'error': str(e)}
            status = rec.get('status', 'error')
            if status == 'ok':
                n_ok += 1
                save_json(rec, docs_dir)
                ok_stats.append({'doc_id': rec['doc_id'], 'status': 'ok', **rec['stats']})
            else:
                if status == 'empty': n_em += 1
                else:                 n_er += 1
                with open(errs, 'a', encoding='utf-8') as ef:
                    ef.write(json.dumps({'doc_id': rec['doc_id'], 'filename': rec.get('filename'),
                                         'status': status, 'reason': rec.get('error')}) + '\n')
            save_checkpoint(ckpt, rec['doc_id'])
            pbar.set_postfix(ok=n_ok, empty=n_em, err=n_er)
        pbar.close()

    elapsed = time.time() - t0
    agg = compute_aggregate(ok_stats, n_ok, n_em, n_er, elapsed)
    with open(stats_f, 'w', encoding='utf-8') as f:
        json.dump(agg, f, indent=2)

    log.info('=' * 60)
    log.info('  DONE')
    log.info('=' * 60)
    log.info(f'  Successful    : {n_ok:,}')
    log.info(f'  Empty/short   : {n_em:,}')
    log.info(f'  Errors        : {n_er:,}')
    log.info(f'  Time          : {elapsed/60:.1f} min  ({agg["docs_per_second"]} docs/sec)')
    if 'tokens' in agg:
        log.info(f'  Total tokens  : {agg["tokens"]["total_corpus"]:,}')
        log.info(f'  Mean tok/doc  : {agg["tokens"]["mean_per_doc"]:,}')
    log.info(f'  JSON files    : {docs_dir}')
    log.info(f'  Stats         : {stats_f}')


if __name__ == '__main__':
    p = argparse.ArgumentParser()
    p.add_argument('--input_dir',  required=True)
    p.add_argument('--output_dir', required=True)
    p.add_argument('--workers',    type=int, default=2)
    p.add_argument('--max_docs',   type=int, default=None)
    a = p.parse_args()
    run_preprocessing(a.input_dir, a.output_dir, a.workers, a.max_docs)

Overwriting /content/preprocess_dataset.py


In [9]:
# ── CELL 5 ── Confirm script was written
import os
path = '/content/preprocess_dataset.py'
if os.path.exists(path):
    lines = open(path).readlines()
    print(f'✓ preprocess_dataset.py — {len(lines)} lines — OK')
    # Confirm the fix is present
    content = open(path).read()
    if 'suffix.lower()' in content:
        print('✓ Case-insensitive PDF fix is present')
    else:
        print('✗ Fix NOT found — re-run Cell 4')
else:
    print('✗ ERROR: file not found — re-run Cell 4')

✓ preprocess_dataset.py — 364 lines — OK
✓ Case-insensitive PDF fix is present


In [10]:
# ── CELL 6 ── Clear old checkpoint + test on 5 docs
import os, json, glob

# Clear checkpoint so we start fresh
ckpt = f'{OUTPUT_DIR}/checkpoint.txt'
if os.path.exists(ckpt):
    os.remove(ckpt)
    print('✓ Old checkpoint cleared')
else:
    print('✓ No old checkpoint')

TEST_OUT = '/content/test_out'

!python /content/preprocess_dataset.py \
    --input_dir  "{INPUT_DIR}" \
    --output_dir "{TEST_OUT}" \
    --workers    2 \
    --max_docs   5

files = sorted(glob.glob(f'{TEST_OUT}/docs/*.json'))
print(f'\n{len(files)} JSON files created')
if files:
    with open(files[0]) as f:
        d = json.load(f)
    st = d['stats']
    print(f"status         : {d['status']}")
    print(f"filename       : {d['filename']}")
    print(f"word_count     : {st['word_count']:,}")
    print(f"total_tokens   : {st['total_tokens']:,}")
    print(f"num_sentences  : {st['num_sentences']}")
    print(f"sections_found : {st['sections_found']}")
    print(f"entity_counts  : {st['entity_counts']}")
else:
    print('⚠ No output — check errors above')

✓ No old checkpoint
15:50:02  ============================================================
15:50:02    LEGAL DATASET PREPROCESSOR
15:50:02  ============================================================
15:50:02    Input   : /content/drive/MyDrive/supreme_court_judgments
15:50:02    Output  : /content/test_out
15:50:02    Workers : 2
15:50:04    PDFs found : 26,703
15:50:04    Limited to : 5
15:50:04    Checkpoint: 5 docs already done — skipping
15:50:04    To process : 0
15:50:04  ============================================================
15:50:04  All documents already processed.

5 JSON files created
status         : ok
filename       : A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_India__on_19_May_1950_1.PDF
word_count     : 94,311
total_tokens   : 94,126
num_sentences  : 2610
sections_found : ['PREAMBLE', 'JUDGMENT']
entity_counts  : {'SECTION': 108, 'ACT': 35, 'PARTY': 5, 'JUDGE': 18, 'LEGAL_TERM': 7}


In [11]:
# ── CELL 7 ── FULL RUN — all 26,703 PDFs
#
# ✅ Resumable  — if Colab disconnects, just re-run this cell only.
#                 It skips already-done docs automatically.
# ⏱ Time       — ~90–150 min on free Colab (2 workers)
#                 ~45–70 min on Colab Pro  (set NUM_WORKERS=4 in Cell 3)
# 🚫 Do NOT close the browser tab while running.

# ── CELL 7 ── Process first 5,000 PDFs only

!python /content/preprocess_dataset.py \
    --input_dir  "{INPUT_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --workers    {NUM_WORKERS} \
    --max_docs   5000

15:50:11  ============================================================
15:50:11    LEGAL DATASET PREPROCESSOR
15:50:11  ============================================================
15:50:11    Input   : /content/drive/MyDrive/supreme_court_judgments
15:50:11    Output  : /content/drive/MyDrive/L_P
15:50:11    Workers : 2
15:50:13    PDFs found : 26,703
15:50:13    Limited to : 5000
15:50:13    To process : 5,000
15:50:13  ============================================================
Preprocessing: 100% 5000/5000 [1:37:56<00:00,  1.18s/doc, empty=0, err=0, ok=5000]
17:28:10  ============================================================
17:28:10    DONE
17:28:10  ============================================================
17:28:10    Successful    : 5,000
17:28:10    Empty/short   : 0
17:28:10    Errors        : 0
17:28:10    Time          : 98.0 min  (0.85 docs/sec)
17:28:10    Total tokens  : 25,657,700
17:28:10    Mean tok/doc  : 5,132
17:28:10    JSON files    : /content/drive/MyDrive

In [12]:
# ── CELL 8 ── Stats report after full run
import json, os

json_count = len([f for f in os.listdir(f'{OUTPUT_DIR}/docs') if f.endswith('.json')])
print(f'JSON files in L_P/docs : {json_count:,}')
print()

with open(f'{OUTPUT_DIR}/stats.json') as f:
    s = json.load(f)

print(f"Successful        : {s['successful']:,}")
print(f"Empty / short     : {s['empty_or_short']:,}")
print(f"Errors            : {s['errors']:,}")
print(f"Time              : {s['processing_time_seconds']/60:.1f} min")
print(f"Speed             : {s['docs_per_second']} docs/sec")
print()
tok = s['tokens']
print(f"Total tokens      : {tok['total_corpus']:,}")
print(f"Mean tokens/doc   : {tok['mean_per_doc']:,}")
print()
print('Section coverage:')
for sec, cov in s['section_coverage'].items():
    bar = '█' * int(cov['percent'] / 5)
    print(f"  {sec:<12}: {cov['percent']:5.1f}%  {bar}")

JSON files in L_P/docs : 5,000

Successful        : 5,000
Empty / short     : 0
Errors            : 0
Time              : 98.0 min
Speed             : 0.85 docs/sec

Total tokens      : 25,657,700
Mean tokens/doc   : 5,132

Section coverage:
  FACTS       :   0.0%  
  ISSUES      :   0.0%  
  ARGUMENTS   :   0.0%  
  REASONING   :   0.1%  
  JUDGMENT    :  99.7%  ███████████████████
  STATUTES    :   0.1%  


In [13]:
# ── CELL 9 ── Inspect a random output JSON
import json, os, random

docs_dir  = f'{OUTPUT_DIR}/docs'
all_files = [f for f in os.listdir(docs_dir) if f.endswith('.json')]
print(f'Total JSON files : {len(all_files):,}')

with open(os.path.join(docs_dir, random.choice(all_files))) as f:
    d = json.load(f)

st = d['stats']
print('=' * 55)
print(f"doc_id         : {d['doc_id']}")
print(f"filename       : {d['filename']}")
print(f"word_count     : {st['word_count']:,}")
print(f"total_tokens   : {st['total_tokens']:,}")
print(f"num_sentences  : {st['num_sentences']}")
print(f"sections_found : {st['sections_found']}")
print(f"has_facts      : {st['has_facts']}")
print(f"has_reasoning  : {st['has_reasoning']}")
print(f"has_judgment   : {st['has_judgment']}")
print(f"entity_counts  : {st['entity_counts']}")
print('=' * 55)
print('\nFirst 5 sentences:')
for s in d['sentences'][:5]:
    print(f"  [{s['section']:<12}] tok={s['tokens']:3d}  {s['text'][:80]}")

Total JSON files : 5,000
doc_id         : doc_00151
filename       : Anglo_French_Textile_Co_Ltd_vs_Commissioner_Of_Income_Tax_Madras_on_22_December_1952 (2)_1.PDF
word_count     : 2,029
total_tokens   : 2,006
num_sentences  : 61
sections_found : ['PREAMBLE', 'JUDGMENT']
has_facts      : False
has_reasoning  : False
has_judgment   : True
entity_counts  : {'SECTION': 8, 'PARTY': 3}

First 5 sentences:
  [PREAMBLE    ] tok= 45  Anglo-French Textile Co., Ltd vs Commissioner Of Income-Tax, Madras on 22 Decemb
  [PREAMBLE    ] tok=  6  Bhagwati PETITIONER: ANGLO-FRENCH TEXTILE CO., LTD.
  [PREAMBLE    ] tok=  6  Vs. RESPONDENT: COMMISSIONER OF INCOME-TAX, MADRAS.
  [PREAMBLE    ] tok= 20  DATE OF JUDGMENT: 22/12/1952 BENCH: MAHAJAN, MEHR CHAND BENCH: MAHAJAN, MEHR CHA
  [PREAMBLE    ] tok= 56  CITATION: 1953 AIR 105 1953 SCR 454 CITATOR INFO : C 1954 SC 198 (10,10A) R 1958
